# The reduced model end to end

`ac.build_reduced_model` runs the whole pipeline of the previous tutorials in one call:
sample, estimate the subspace, choose $n$, build the active domain and fit a surrogate
on the active variables. The result, an `ac.ReducedModel`, predicts $f$ and answers
three questions: the **average** of $f$, the **probability** that $f$ falls in an
interval, and the **minimum** of $f$. See {doc}`../user_guide/reduced_model` and
{doc}`../theory/reduced_model`.

The example is the output transformerless push-pull (OTL) circuit, a built-in test
function of six inputs whose output is a midpoint voltage.

In [ ]:
%matplotlib inline
import tempfile
from pathlib import Path

import numpy as np

import active_subspaces as ac

otl = ac.test_functions.OTL_CIRCUIT
print(otl.description, "| inputs:", otl.space.names, "| output in", otl.qoi_units)

## Build from a model

Given a model, the builder evaluates values and gradients at random inputs (by default
$\lfloor 6(m+1)\ln m\rfloor$ points), estimates the subspace with a bootstrap,
chooses $n$, and fits an RBF surrogate to averages of $f$ over the inactive variables at
design points in the active domain. `r_squared` is a holdout $R^2$: it compares the
surrogate with the sampled values, which the surrogate fit did not use.

In [ ]:
rm = ac.build_reduced_model(model=otl, rng=1)
print(f"n = {rm.n}, {rm.domain.kind} domain, {len(rm.f)} samples")
print(f"R^2 ({rm.r_squared_kind}) = {rm.r_squared:.3f}")

The subspace, domain and surrogate are fields of the reduced model, so every plot of
the earlier tutorials applies:

In [ ]:
fig, ax = ac.plotting.plot_eigenvalues(rm.subspace)
fig

## Predict

`ac.predict(rm, X)` (or `rm.predict(X)`) takes physical inputs. On new points:

In [ ]:
X_test = otl.space.sample(500, rng=2)
f_test = ac.evaluate(otl, X=X_test)
f_hat = ac.predict(rm, X_test)
r2 = 1 - np.sum((f_test - f_hat) ** 2) / np.sum((f_test - f_test.mean()) ** 2)
print(f"test R^2 = {r2:.3f}, largest error {np.max(abs(f_test - f_hat)):.3f} V")
fig, ax = ac.plotting.plot_surrogate_fit(rm.surrogate, X=X_test, f=f_test)
fig

## Average

`rm.average` integrates the surrogate over the active domain, with no model
evaluations. With `model=` it integrates $f$ itself, averaging `n_mc` model evaluations
(default 50) over each fiber, and reports the standard error of those averages.

In [ ]:
avg_s = rm.average(n_nodes=7)
avg_f = rm.average(n_nodes=7, model=otl, rng=3)
ref = otl.reference_results
print(f"surrogate: {avg_s.estimate:.4f}")
print(f"model:     {avg_f.estimate:.4f} +/- {avg_f.inactive_std_error:.4f}")
print(f"reference: {ref.mean:.4f} +/- {ref.mean_se:.4f}")

## Probability

`rm.probability(lower, upper)` estimates $P[\text{lower} \le g(W_1^\top x) \le
\text{upper}]$ by Monte Carlo on the **surrogate** (10 000 draws by default), and
`confidence_interval` gives a Wilson score interval (Wilson 1927) for the Monte Carlo
error. The surrogate averages $f$ over the inactive variables, which smooths it, so
this probability can differ from that of $f$ by more than the Monte Carlo error. The
OTL circuit is cheap, so compare with plain Monte Carlo on the model:

In [ ]:
pr = rm.probability(-np.inf, 5.0, rng=4)
lo, hi = pr.confidence_interval(0.99)
print(f"P[V_m <= 5 V], surrogate: {pr.p:.4f}, 99% interval [{lo:.4f}, {hi:.4f}]")
X_mc = otl.space.sample(10_000, rng=5)
print(f"P[V_m <= 5 V], model:     {np.mean(ac.evaluate(otl, X=X_mc) <= 5.0):.4f}")

## Minimum

`rm.minimum` minimizes the surrogate over the active domain and lifts the minimizer to
a full input, as `ac.minimize` does (see {doc}`exploiting_the_subspace`, including the
`accuracy.curvature_clipped` warning). With `model=` it also evaluates $f$ there.

In [ ]:
mn = rm.minimum(model=otl, rng=6)
print(f"g(y*) = {mn.f_star:.4f} V, f(x*) = {mn.f_model:.4f} V")
names = otl.space.names
print("x* =", "  ".join(f"{k}={v:.4g}" for k, v in zip(names, mn.x_star_physical, strict=True)))

## Build from data

With data instead of a model, pass `X`, `f` and the space (and `grads` if you have
them). Without gradients the builder estimates them with local linear fits, and
`r_squared` is a leave-one-out $R^2$.

In [ ]:
X_data = otl.space.sample(300, rng=7)
f_data = ac.evaluate(otl, X=X_data)
rm_data = ac.build_reduced_model(X=X_data, f=f_data, space=otl.space, rng=8)
print(
    f"n = {rm_data.n}; gradients: {rm_data.gradient_source}; "
    f"R^2 ({rm_data.r_squared_kind}) = {rm_data.r_squared:.3f}"
)

## When the reduced model does not fit

Not every function has a useful active subspace. The robot arm function's eigenvalues
decay slowly, and the $R^2$ of its reduced model is near zero or negative: the reduced
model predicts no better than the mean. Always check `r_squared` before using the
answers.

In [ ]:
robot = ac.test_functions.ROBOT_ARM
rm_robot = ac.build_reduced_model(model=robot, rng=9)
print("eigenvalues:", np.array2string(rm_robot.eigenvalues, precision=3))
print(f"n = {rm_robot.n}, R^2 = {rm_robot.r_squared:.3f}")

## Save and reload

A reduced model is a result like any other: it saves to `.npz` and reloads with
`ac.load`, and the reloaded copy predicts without the original model in memory.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "otl_reduced_model.npz"
    rm.save(path)
    rm_back = ac.load(path)
print(
    "equal:",
    rm_back == rm,
    "| same predictions:",
    np.array_equal(ac.predict(rm_back, X_test), f_hat),
)

## References

- Constantine, P. G. (2015). *Active Subspaces: Emerging Ideas for Dimension Reduction
  in Parameter Studies.* SIAM.
- Wilson, E. B. (1927). Probable inference, the law of succession, and statistical
  inference. *Journal of the American Statistical Association*, 22(158), 209–212.